# Advanced 10 — Long-Running & Asynchronous Agents

A long-running agent is a **durable, versioned state machine**. It must survive process death, duplicate delivery, stale authority, competing workers, and uncertain external effects. A callback can wake the workflow; it cannot authorize execution by itself.

## Learning path

1. Inspect the typed state machine and persisted records.
2. Prove that process B can resume state written by process A.
3. Admit and deduplicate signed events.
4. Bind approval to one proposal and current preconditions.
5. Resolve timer, cancellation, and worker races.
6. Claim a stable logical operation, call outside the transaction, and reconcile an unknown outcome.
7. Compare the governed runtime with a same-task naive baseline.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
from datetime import timedelta
import json
import sys

COURSE_DIR = Path.cwd() / 'curriculum' / 'advanced' / '10-long-running-asynchronous-agents'
sys.path.insert(0, str(COURSE_DIR))

from policy import EventType, RunStatus, TimeoutPolicy
from lab import (
    FIXED_TIME, DurableStore, DeterministicProvider, EventDisposition,
    OperationDisposition, ProviderOutcome, WorkflowRuntime,
    evaluate_same_cases, fixture_approval, fixture_context,
    fixture_preconditions, happy_path_demo, signed_event,
)

workspace = TemporaryDirectory()
WORK_DIR = Path(workspace.name)
print(f'credential-free workspace: {WORK_DIR.name}')

## 1. State and storage are application-owned

`RunStatus` separates waiting, readiness, execution, reconciliation, verification, and terminal outcomes. The SQLite adapter persists the current run plus inbox, outbox, history, approvals, timers, logical operations, and attempt receipts. It stores authoritative execution data—not an entire prompt or arbitrary in-process objects.

Every transition uses a legal-transition check and a compare-and-swap `state_version`. The lab never uses `INSERT OR REPLACE`, because replacement can hide concurrent mutation.

In [ ]:
result = happy_path_demo(WORK_DIR / 'restart.db')
print(json.dumps(result, indent=2))
assert result['status'] == 'COMPLETED'
assert result['provider_calls'] == 1

The demo deliberately opens the same file through two separate `DurableStore`/`WorkflowRuntime` instances. Process-local memory is not the checkpoint. Completion occurs only after the provider effect is independently queried and matched to the request digest.

## 2. Event delivery is not authority

The event envelope binds event ID, source, tenant, run, type, timestamp, payload digest, and signature. The durable inbox deduplicates delivery. An approval event only points to a separately stored, typed, single-use receipt.

In [ ]:
runtime = WorkflowRuntime(DurableStore(WORK_DIR / 'dedupe.db'))
waiting = runtime.start_approval_run('run-dedupe')
receipt = fixture_approval(waiting)
runtime.store.put_approval(receipt)
event = signed_event(
    event_id='event-approval', run_id=waiting.run_id,
    event_type=EventType.APPROVAL_AVAILABLE,
    payload={'approval_id': receipt.approval_id},
    occurred_at=FIXED_TIME + timedelta(minutes=6),
)
first = runtime.process_event(event, fixture_context(), now=FIXED_TIME + timedelta(minutes=6))
duplicate = runtime.process_event(event, fixture_context(), now=FIXED_TIME + timedelta(minutes=7))
print(first.disposition, duplicate.disposition, runtime.store.inbox_count(event.event_id))
assert first.disposition is EventDisposition.PROCESSED
assert duplicate.disposition is EventDisposition.DUPLICATE
assert first.state_version == duplicate.state_version

## 3. Approval is bound and revalidated

A receipt binds tenant, subject, proposal ID/digest, precondition digest, action, target, approver role, policy version, and expiry. Execution also checks current policy, current role, permitted action/target, and the current precondition digest. Approval of `order-v7` cannot authorize a changed `order-v8`.

In [ ]:
changed_runtime = WorkflowRuntime(DurableStore(WORK_DIR / 'changed.db'))
changed_waiting = changed_runtime.start_approval_run('run-changed')
changed_receipt = fixture_approval(changed_waiting)
changed_runtime.store.put_approval(changed_receipt)
changed_event = signed_event(
    event_id='event-changed', run_id=changed_waiting.run_id,
    event_type=EventType.APPROVAL_AVAILABLE,
    payload={'approval_id': changed_receipt.approval_id},
    occurred_at=FIXED_TIME + timedelta(minutes=6),
)
changed_context = fixture_context(
    preconditions=fixture_preconditions(resource_version='order-v8')
)
rejected = changed_runtime.process_event(
    changed_event, changed_context, now=FIXED_TIME + timedelta(minutes=6)
)
print(rejected.reason_codes)
assert rejected.disposition is EventDisposition.REJECTED
assert 'PRECONDITION_CHANGED' in rejected.reason_codes

## 4. Durable timers and races

Waiting releases the worker. A persisted timer later applies the stored timeout policy. Approval and timeout both attempt a legal transition from the current version; the first committed transition wins and the other delivery becomes stale. Long process-local sleeps are the anti-pattern. Polling itself is not universally forbidden—managed systems may poll durably.

In [ ]:
race_runtime = WorkflowRuntime(DurableStore(WORK_DIR / 'race.db'))
race_waiting = race_runtime.start_approval_run(
    'run-race', timeout_after=timedelta(minutes=10),
    timeout_policy=TimeoutPolicy.EXPIRE,
)
race_receipt = fixture_approval(race_waiting)
race_runtime.store.put_approval(race_receipt)
approval = signed_event(
    event_id='race-approval', run_id=race_waiting.run_id,
    event_type=EventType.APPROVAL_AVAILABLE,
    payload={'approval_id': race_receipt.approval_id},
    occurred_at=FIXED_TIME + timedelta(minutes=6),
)
approved = race_runtime.process_event(approval, fixture_context(), now=FIXED_TIME + timedelta(minutes=6))
timer = signed_event(
    event_id='race-timer', run_id=race_waiting.run_id,
    event_type=EventType.TIMER_FIRED,
    payload={'timer_id': race_waiting.pending_timer_id},
    occurred_at=FIXED_TIME + timedelta(minutes=10),
)
late_timer = race_runtime.process_event(timer, fixture_context(), now=FIXED_TIME + timedelta(minutes=10))
print(approved.run_status, late_timer.disposition)
assert approved.run_status is RunStatus.READY_TO_RESUME
assert late_timer.disposition is EventDisposition.STALE

## 5. Three idempotency boundaries

- `event_id` deduplicates message delivery.
- `state_version` serializes state transitions.
- `logical_operation_id + request_digest` identifies an external effect.

The worker first commits an operation claim and unique attempt receipt, then calls the provider outside the database transaction, then records the result. The logical ID remains stable across retries; attempt IDs are unique. This supports at-least-once delivery without claiming universal exactly-once execution.

## 6. Unknown outcome requires reconciliation

A timeout after a possible provider commit is not a transient failure. The run enters `RECONCILING`, and another execution claim returns `RECONCILE_REQUIRED`. The provider is queried by the stable operation ID before any retry.

In [ ]:
unknown_runtime = WorkflowRuntime(DurableStore(WORK_DIR / 'unknown.db'))
unknown_waiting = unknown_runtime.start_approval_run('run-unknown')
unknown_receipt = fixture_approval(unknown_waiting)
unknown_runtime.store.put_approval(unknown_receipt)
unknown_event = signed_event(
    event_id='unknown-approval', run_id=unknown_waiting.run_id,
    event_type=EventType.APPROVAL_AVAILABLE,
    payload={'approval_id': unknown_receipt.approval_id},
    occurred_at=FIXED_TIME + timedelta(minutes=6),
)
unknown_runtime.process_event(unknown_event, fixture_context(), now=FIXED_TIME + timedelta(minutes=6))
ready = unknown_runtime.store.load_run(unknown_waiting.run_id)
lease = unknown_runtime.store.claim_lease(
    ready.run_id, 'worker-a', expected_version=ready.state_version,
    now=FIXED_TIME + timedelta(minutes=7),
)
prepared = unknown_runtime.store.prepare_operation(
    ready.run_id, 'worker-a', fixture_context(),
    now=FIXED_TIME + timedelta(minutes=7, seconds=1),
)
provider = DeterministicProvider()
uncertain = unknown_runtime.execute(
    prepared, provider, ProviderOutcome.TIMEOUT_AFTER_COMMIT,
    now=FIXED_TIME + timedelta(minutes=8),
)
blocked = unknown_runtime.store.prepare_operation(
    ready.run_id, 'worker-a', fixture_context(),
    now=FIXED_TIME + timedelta(minutes=8, seconds=1),
)
reconciled = unknown_runtime.reconcile(
    prepared.logical_operation_id, provider,
    now=FIXED_TIME + timedelta(minutes=9),
)
print(uncertain.status, blocked.disposition, reconciled.status, provider.calls)
assert uncertain.status is RunStatus.RECONCILING
assert blocked.disposition is OperationDisposition.RECONCILE_REQUIRED
assert reconciled.status is RunStatus.VERIFYING
assert provider.calls == 1

## 7. Same-task evaluation

The naive baseline and governed runtime receive the same four failure cases: duplicate delivery, cross-tenant event, cancel-then-late-approval, and timeout-after-commit. Deterministic replay measures control behavior—not database scale, live provider reliability, or model intelligence.

In [ ]:
report = evaluate_same_cases(WORK_DIR / 'evaluation')
rows = {
    'baseline': report.baseline.model_dump() | {'safe_outcome_rate': report.baseline.safe_outcome_rate},
    'governed': report.governed.model_dump() | {'safe_outcome_rate': report.governed.safe_outcome_rate},
}
print(json.dumps(rows, indent=2))
assert report.baseline.unsafe_resumes == 2
assert report.baseline.duplicate_effects == 2
assert report.governed.safe_outcome_rate == 1.0
assert report.governed.provider_calls == 2

## Production translation

Temporal, LangGraph, AWS Step Functions, Azure Durable Functions, or a database-backed worker system can supply persistence, histories, timers, and task delivery. They do not decide what an approval means, which tenant owns a run, whether preconditions still hold, how a remote effect is reconciled, or what evidence permits completion.

Before production: choose a durable orchestrator/store, implement authenticated event ingress and key rotation, use a provider-supported idempotency/reconciliation contract, encrypt and retain state under policy, test concurrent workers and crash windows, publish the outbox, migrate versioned histories safely, and monitor stuck waits, leases, budgets, and reconciliations.

## Knowledge check

1. Why are an event ID, state version, and logical operation ID all necessary?
2. Why does a valid approval event still require current-policy and precondition checks?
3. What must happen after a timeout when the provider may have committed?
4. Why is a durable timer different from `sleep()` in a worker process?
5. Which responsibilities remain application-owned when using a managed workflow engine?